# Clustering Rough Work — Experiments, Decisions, and Superseded Approaches

This notebook is the **research log** for Airbnb listing segmentation. It keeps the experiments, diagnostics, alternatives, results, and rationale used to select a clustering approach.

Use this notebook to understand *why* choices were made. The separate `clustering.ipynb` is the concise pipeline for regenerating the two selected output CSVs.

**Input:** `../data/ml/ml_ready_dataset.csv`  
**Current candidate:** K-Means, `k = 2`, without latitude/longitude, retaining neighbourhood-group indicators and host portfolio size.  
**Outputs:** `../data/ml/outputs/clustered_listings_kmeans_k2.csv` and `../data/ml/outputs/cluster_profile_kmeans_k2.csv`.

> Run from the `notebooks/` directory. This rough-work notebook has been reorganized to avoid duplicate cells and stale notebook state. Previous results are summarized below; rerunning cells will regenerate results from the current input file.

## Current decision snapshot

The latest experiments support the following *candidate*, not a claim that a production-ready model has been proven:

- Cap `minimum nights` at 365, then apply `log1p` to `minimum nights`, `number of reviews`, and `calculated host listings count`.
- Exclude `service fee` from clustering because it correlates strongly with `price` (previously measured Pearson \(r \approx 0.75\)).
- Exclude latitude/longitude from the selected feature matrix because neighbourhood-group indicators already represent location at a broader level.
- Scale continuous features with `RobustScaler`; leave binary and one-hot columns as 0/1.
- Fit K-Means with `k=2`, `random_state=42`, and `n_init=10`.

Latest comparison results:

| Feature configuration | k | Silhouette | Smallest cluster | Largest cluster |
|---|---:|---:|---:|---:|
| Current baseline (coordinates included) | 2 | 0.3500 | 8.48% | 91.52% |
| **Without coordinates** | **2** | **0.3912** | **8.48%** | **91.52%** |
| Without host portfolio size | 2 | 0.1682 | 27.40% | 72.60% |

For `k=2`, baseline vs. no-coordinates labels had ARI ≈ 0.983, meaning they were almost the same partition. The no-coordinate model was also highly stable across the tested random seeds (ARI ≈ 0.999–1.000).

The final profiles show a major distinction between a small cluster of portfolio-scale hosts with longer minimum stays and high availability, and a large cluster of smaller host portfolios with shorter minimum stays and higher review activity. Mean prices are similar, so this is **not primarily a price-tier segmentation**.

## Notebook map and why some earlier cells are not carried into the final pipeline

| Experiment / code | Why it was used | Decision |
|---|---|---|
| Data quality checks and correlation heatmap | Check value ranges, feature spread, and redundant signals before distance-based clustering. | Retained here as diagnostic work. |
| Initial model with capped but untransformed minimum nights | Establish a baseline and reveal the effect of extreme long-stay values. | Kept as a historical experiment; not selected. |
| Log-transforming selected skewed features | Reduce extreme scale effects while preserving the ordering of values. | Retained in the selected candidate. |
| `service fee` | Highly correlated with price, potentially double-weighting price information. | Excluded from clustering features; original values remain in the listing-level output. |
| Latitude and longitude | Test whether exact coordinates add value beyond neighbourhood-group indicators. | Excluded in the selected candidate after ablation; retained in the original output data. |
| Host portfolio size | Test whether host portfolio size dominates clustering. | An alternative without it was tested; retained in the candidate because separation dropped substantially when removed. |
| k = 3 and k = 4 | Explore more detailed segments. | Kept as comparisons, not selected because silhouette was lower and at least one cluster was weakly separated. |
| PCA | Visualize a high-dimensional feature matrix in two dimensions and diagnose dominant variation. | Visualization/diagnostic only; PCA components are not used to fit K-Means. |
| Raw `house_rules` | Potential source of NLP-derived information. | Excluded from this structured K-Means pipeline; keep for a separate NLP experiment. |
| Older duplicate profile/silhouette/PCA cells | Some used `X_scaled` or `cluster_labels` from earlier notebook execution states. | Rewritten below in execution order to avoid accidentally reporting results from a stale feature set. |
| Hierarchical clustering / DBSCAN | Possible alternative algorithms. | Not implemented yet; could be considered later if K-Means results do not serve the objective. |

**Important interpretation note:** the very high early silhouette score (~0.934 at `k=2`) came with an extremely imbalanced solution (about 99.74% vs. 0.26%). It was treated as an unhelpful first baseline rather than evidence that the final model should use two clusters without further validation.

## 1. Load and inspect the ML-ready dataset

This cell establishes the input shape, confirms that the exported features are numeric and checks missing values/duplicates. It is kept because every downstream experiment must use the same known input snapshot.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_samples, silhouette_score
from sklearn.preprocessing import RobustScaler

# This relative path assumes the notebook is run from the project's notebooks/ folder.
DATA_PATH = Path("../data/ml/ml_ready_dataset.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. "
        "Run this notebook from the notebooks/ directory or update DATA_PATH."
    )

# Load the structured, encoded feature dataset.
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("\nData types:")
display(df.dtypes.value_counts().rename("column_count").to_frame())

# Basic checks: downstream K-Means expects a complete numeric feature matrix.
if df.isna().any().any():
    raise ValueError("The ML-ready dataset contains missing values. Resolve them before clustering.")
non_numeric = df.select_dtypes(exclude=np.number).columns.tolist()
if non_numeric:
    raise ValueError(f"Unexpected non-numeric columns in the structured dataset: {non_numeric}")

display(df.head())

## 2. Data quality, range checks, and correlation diagnostics

These checks were used to revisit known risks before modeling: extreme minimum-night values, feature scales with very different ranges, and the price/service-fee relationship. The correlation heatmap is exploratory; it does not alone decide which features to include.

In [ ]:
# Summarize numerical ranges, unique values, and spread before transformations.
quality_summary = pd.DataFrame({
    "min": df.min(numeric_only=True),
    "25%": df.quantile(0.25, numeric_only=True),
    "median": df.median(numeric_only=True),
    "75%": df.quantile(0.75, numeric_only=True),
    "max": df.max(numeric_only=True),
    "unique_values": df.nunique(),
    "std": df.std(numeric_only=True)
})
display(quality_summary.sort_values("std", ascending=False))

# Print constraints that matter for the known Airbnb features.
for col in ["minimum nights", "availability 365", "price", "service fee"]:
    if col in df.columns:
        print(f"\n{col}")
        print("min:", df[col].min(), "| max:", df[col].max())
        print("negative values:", int((df[col] < 0).sum()))

# Quantify the known relationship rather than relying only on a heatmap.
if {"price", "service fee"}.issubset(df.columns):
    print(
        "\nPearson correlation: price vs service fee =",
        round(df["price"].corr(df["service fee"]), 4)
    )

# Plot a correlation matrix for the encoded numerical features.
corr = df.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(14, 10))
im = ax.imshow(corr, aspect="auto", vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticks(range(len(corr.columns)))
ax.set_yticklabels(corr.columns)
fig.colorbar(im, ax=ax, label="Pearson correlation")
ax.set_title("Correlation Matrix — ML-ready Features")
plt.tight_layout()
plt.show()

## 3. Historical baseline: cap minimum nights, but do not log-transform yet

### Why this experiment was run
The first clustering attempt capped `minimum nights` at 365 but left the long tail otherwise untransformed. It was useful as a baseline because it showed that a very high silhouette score can be misleading when K-Means isolates a tiny extreme-value group.

Historical outputs recorded during this experiment:
- `k=2` silhouette ≈ **0.9339**; clusters ≈ **99.74% / 0.26%**.
- `k=3` silhouette ≈ 0.5258; clusters ≈ 14.83% / 84.91% / 0.26%.
- `k=4` silhouette ≈ 0.5075; clusters ≈ 84.84% / 0.14% / 0.51% / 14.52%.
- PCA's first two components explained ≈ 76.57%, and PC1 was almost entirely driven by `minimum nights` (loading ≈ 0.996).

### Why it was not selected
The 0.9339 score accompanied a cluster of only about 164 listings out of 62,605. This was consistent with the model separating a small group of extreme minimum-stay listings rather than finding balanced, broadly useful segments.

The code below reproduces this *historical methodology*. Its silhouette score is calculated on a separately fitted 5,000-row sample, whereas the later feature-ablation experiment evaluates sampled rows using labels from a model fitted on the full dataset. Therefore, scores across these two evaluation procedures should not be compared as if they were perfectly identical metrics.

In [ ]:
# Create a separate copy so the source input remains unchanged.
df_initial = df.copy()

# Apply the earlier cap, but deliberately do not apply log1p here.
df_initial["minimum nights"] = df_initial["minimum nights"].clip(upper=365)

# The initial baseline excluded service fee, but retained coordinates.
X_initial = df_initial.drop(columns=["service fee"], errors="ignore")

# Preserve binary and one-hot encoded columns as 0/1; scale continuous features.
initial_binary_cols = [
    col for col in X_initial.columns
    if X_initial[col].nunique() <= 2
    and set(X_initial[col].dropna().unique()).issubset({0, 1})
]
initial_continuous_cols = [
    col for col in X_initial.columns if col not in initial_binary_cols
]

initial_scaler = RobustScaler()
X_initial_scaled = X_initial.copy()
X_initial_scaled[initial_continuous_cols] = initial_scaler.fit_transform(
    X_initial[initial_continuous_cols]
)

print("Historical baseline feature matrix:", X_initial_scaled.shape)

In [ ]:
# Test the earlier k=2..10 range.
# Inertia comes from a model fit on all rows; for historical reproducibility,
# silhouette is calculated on a separately fitted, reproducible 5,000-row sample.
k_values_initial = range(2, 11)
initial_sample_idx = X_initial_scaled.sample(
    n=min(5000, len(X_initial_scaled)),
    random_state=42
).index
X_initial_sample = X_initial_scaled.loc[initial_sample_idx]

initial_results = []
initial_full_labels = {}

for k in k_values_initial:
    # Full-dataset fit gives the inertia and the full-data cluster-size distribution.
    full_model = KMeans(n_clusters=k, random_state=42, n_init=10)
    full_labels = full_model.fit_predict(X_initial_scaled)
    initial_full_labels[k] = full_labels

    # Historical silhouette procedure: fit a separate model on the sample.
    sample_model = KMeans(n_clusters=k, random_state=42, n_init=10)
    sample_labels = sample_model.fit_predict(X_initial_sample)
    sample_silhouette = silhouette_score(X_initial_sample, sample_labels)

    initial_results.append({
        "k": k,
        "inertia_full_data": full_model.inertia_,
        "silhouette_sample_model": sample_silhouette
    })

initial_results_df = pd.DataFrame(initial_results)
display(initial_results_df.round(4))

# Plot historical inertia and silhouette curves for diagnostic comparison.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(initial_results_df["k"], initial_results_df["inertia_full_data"], marker="o")
axes[0].set_title("Historical Baseline — Elbow")
axes[0].set_xlabel("Number of clusters (k)")
axes[0].set_ylabel("Inertia")

axes[1].plot(
    initial_results_df["k"],
    initial_results_df["silhouette_sample_model"],
    marker="o"
)
axes[1].set_title("Historical Baseline — Silhouette on Sample-Fitted Models")
axes[1].set_xlabel("Number of clusters (k)")
axes[1].set_ylabel("Silhouette score")
plt.tight_layout()
plt.show()

# Confirm the imbalance for the earliest candidate models.
initial_size_rows = []
for k in [2, 3, 4]:
    counts = pd.Series(initial_full_labels[k]).value_counts().sort_index()
    for cluster_id, count in counts.items():
        initial_size_rows.append({
            "k": k,
            "cluster": cluster_id,
            "listings": int(count),
            "share_pct": count / len(df) * 100
        })
display(pd.DataFrame(initial_size_rows).round(2))

### 3A. Historical baseline cluster profiles (before log transformations)

This cell profiles the original k=2, 3, and 4 solutions from the untransformed baseline. It was used to see whether the very small cluster(s) represented useful listing segments or a group largely separated by extreme minimum-night values.

**Why it is retained here:** it shows why the early high silhouette score was not accepted as the final model. These profiles are historical diagnostics and do not describe the selected no-coordinate model.

In [ ]:
# Compare cluster sizes and original-unit profiles for the historical baseline.
# Labels were produced by the full-data fits in the preceding baseline experiment.
baseline_profile_data = df_initial.copy()

for k in [2, 3, 4]:
    temp = baseline_profile_data.copy()
    temp["cluster"] = initial_full_labels[k]

    print(f"\nHistorical baseline profiles — k={k}")
    display(
        temp.groupby("cluster").agg(
            listings=("price", "size"),
            price_mean=("price", "mean"),
            minimum_nights_median=("minimum nights", "median"),
            reviews_mean=("number of reviews", "mean"),
            reviews_per_month_mean=("reviews per month", "mean"),
            host_listings_median=("calculated host listings count", "median"),
            availability_mean=("availability 365", "mean")
        ).round(2)
    )

    # One-hot encoded category means are the share of listings in each category.
    for group_name, columns in {
        "Neighbourhood": [c for c in temp.columns if c.startswith("neighbourhood group_")],
        "Room type": [c for c in temp.columns if c.startswith("room type_")],
    }.items():
        if columns:
            print(f"{group_name} distribution (%)")
            display((temp.groupby("cluster")[columns].mean() * 100).round(1))

### 3B. Historical baseline PCA diagnosis

In the first version, the first two PCA components explained about 76.57% of the variance, but PC1 was almost entirely driven by `minimum nights` (absolute loading approximately 0.996). This exposed the problem: the model could separate extreme minimum-stay listings and earn a high silhouette score without providing a balanced segmentation.

The following cell reproduces that diagnostic. PCA is used only to inspect variation and visualize data; its components are not the inputs to K-Means.

In [ ]:
# Fit PCA to the historical scaled matrix before log transformations.
pca_initial = PCA(n_components=2, random_state=42)
X_pca_initial = pca_initial.fit_transform(X_initial_scaled)

print(
    "Historical baseline variance explained by PC1 and PC2:",
    round(pca_initial.explained_variance_ratio_.sum() * 100, 2),
    "%"
)

# Loadings help diagnose which variables dominate each component.
initial_loadings = pd.DataFrame(
    pca_initial.components_.T,
    index=X_initial_scaled.columns,
    columns=["PC1", "PC2"]
)
initial_loadings["abs_PC1"] = initial_loadings["PC1"].abs()
initial_loadings["abs_PC2"] = initial_loadings["PC2"].abs()

print("Top historical PC1 contributors:")
display(initial_loadings.sort_values("abs_PC1", ascending=False).head(10))

print("Top historical PC2 contributors:")
display(initial_loadings.sort_values("abs_PC2", ascending=False).head(10))

# A small visualization provides context; cluster fitting remains in original feature space.
# Use the historical k=3 full-data labels to reproduce the earlier visual diagnostic.
plt.figure(figsize=(10, 7))
scatter = plt.scatter(
    X_pca_initial[:, 0],
    X_pca_initial[:, 1],
    c=initial_full_labels[3],
    s=8,
    alpha=0.45
)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("Historical Baseline PCA Projection (k=3)")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.show()

## 4. Revised feature preparation: cap and log-transform selected skewed features

### Why this was changed
The previous diagnostic showed `minimum nights` had a median of 3, a 99th percentile of 45, and a maximum of 5,645 before capping. After capping alone, the upper tail still strongly influenced PC1. Applying `log1p` compresses the long tail while preserving ordering.

The same transformation is applied to `number of reviews` and `calculated host listings count`, which are also right-skewed counts. The original `df` remains unchanged so profiles can be reported in interpretable original units.

The feature matrix `X` below excludes `service fee` due to redundancy with `price`; coordinates are tested separately in the ablation experiment.

In [ ]:
# Keep modeling transforms in a separate DataFrame.
df_cluster = df.copy()

# Cap unrealistic/extreme minimum-night requirements for this learning project.
# This is a documented modeling rule, not proof that every stay above 365 is invalid.
df_cluster["minimum nights"] = df_cluster["minimum nights"].clip(upper=365)

# Compress skewed, non-negative count features without changing their order.
features_to_transform = [
    "minimum nights",
    "number of reviews",
    "calculated host listings count"
]
for col in features_to_transform:
    if (df_cluster[col] < 0).any():
        raise ValueError(f"{col} contains negative values and cannot be log1p transformed safely.")
    df_cluster[col] = np.log1p(df_cluster[col])

# Exclude service fee to avoid giving correlated pricing variables duplicate influence.
X = df_cluster.drop(columns=["service fee"])

print("Transformed feature matrix:", X.shape)
display(X[features_to_transform].describe().T)

## 5. Feature ablation: test which features are needed

Three configurations were compared:
1. **Current baseline:** transformed features, including latitude/longitude.
2. **Without coordinates:** keep neighbourhood-group indicators but drop exact latitude and longitude.
3. **Without host portfolio size:** drop `calculated host listings count`.

Each feature configuration gets its own `RobustScaler`. Binary and one-hot columns remain 0/1. This experiment tests whether exact coordinates or host portfolio size disproportionately influence the partitions.

The same sample of up to 5,000 row indices is used for silhouette evaluation across all configurations. Inertia is recorded, but **do not compare raw inertia across feature configurations as if it were on an identical scale**, because the number of features differs.

In [ ]:
# Define the alternative feature sets without modifying the shared transformed matrix.
feature_sets = {
    "Current baseline": X.copy(),
    "Without coordinates": X.drop(columns=["lat", "long"], errors="ignore"),
    "Without host portfolio size": X.drop(
        columns=["calculated host listings count"], errors="ignore"
    )
}

scaled_feature_sets = {}
feature_scalers = {}
feature_columns = {}

for variant_name, X_variant in feature_sets.items():
    # Identify binary/one-hot columns that should remain as 0/1.
    binary_cols = [
        col for col in X_variant.columns
        if X_variant[col].nunique() <= 2
        and set(X_variant[col].dropna().unique()).issubset({0, 1})
    ]
    continuous_cols = [
        col for col in X_variant.columns if col not in binary_cols
    ]

    # Fit a separate scaler for this exact feature configuration.
    scaler = RobustScaler()
    X_variant_scaled = X_variant.copy()
    X_variant_scaled[continuous_cols] = scaler.fit_transform(
        X_variant[continuous_cols]
    )

    scaled_feature_sets[variant_name] = X_variant_scaled
    feature_scalers[variant_name] = scaler
    feature_columns[variant_name] = {
        "binary": binary_cols,
        "continuous": continuous_cols
    }

    print(
        f"{variant_name}: rows={X_variant_scaled.shape[0]}, "
        f"features={X_variant_scaled.shape[1]}, "
        f"continuous_scaled={len(continuous_cols)}, "
        f"binary_preserved={len(binary_cols)}"
    )

In [ ]:
# Use one reproducible evaluation sample for all feature-set / k comparisons.
ablation_sample_idx = X.sample(
    n=min(5000, len(X)),
    random_state=42
).index

ablation_results = []
experiment_models = {}
experiment_labels = {}

for variant_name, X_variant_scaled in scaled_feature_sets.items():
    X_eval = X_variant_scaled.loc[ablation_sample_idx]

    for k in [2, 3, 4]:
        # Fit each candidate on the full dataset to obtain final label assignments.
        model = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = model.fit_predict(X_variant_scaled)
        labels_series = pd.Series(labels, index=X_variant_scaled.index)

        # Score the evaluation sample using the full model's assignments.
        labels_eval = labels_series.loc[ablation_sample_idx]
        sample_silhouette = silhouette_score(X_eval, labels_eval)

        counts = labels_series.value_counts()
        ablation_results.append({
            "feature_set": variant_name,
            "k": k,
            "silhouette": sample_silhouette,
            "inertia": model.inertia_,
            "smallest_cluster_pct": counts.min() / len(labels_series) * 100,
            "largest_cluster_pct": counts.max() / len(labels_series) * 100
        })

        experiment_models[(variant_name, k)] = model
        experiment_labels[(variant_name, k)] = labels

ablation_results_df = pd.DataFrame(ablation_results)
display(ablation_results_df.round(4))

### Feature-ablation result

The latest saved results were:

| Feature configuration | k | Silhouette | Smallest cluster | Largest cluster |
|---|---:|---:|---:|---:|
| Current baseline | 2 | 0.3500 | 8.48% | 91.52% |
| Current baseline | 3 | 0.1807 | 8.12% | 66.31% |
| Current baseline | 4 | 0.1243 | 7.58% | 41.65% |
| Without coordinates | 2 | **0.3912** | 8.48% | 91.52% |
| Without coordinates | 3 | 0.1958 | 8.17% | 69.80% |
| Without coordinates | 4 | 0.1207 | 7.71% | 39.33% |
| Without host portfolio size | 2 | 0.1682 | 27.40% | 72.60% |
| Without host portfolio size | 3 | 0.1158 | 17.41% | 43.41% |
| Without host portfolio size | 4 | 0.1278 | 13.45% | 37.74% |

**Decision:** proceed with the no-coordinate configuration as the leading candidate at `k=2`. It improves silhouette over the coordinate-included baseline without changing the main cluster size proportions. Excluding host portfolio size creates more balanced groups, but separation drops substantially; host portfolio size is therefore retained provisionally because it is a relevant business characteristic.

## 6. Compare the k=2 cluster profiles across feature configurations

This is an interpretation step, not another model fit. It profiles each k=2 assignment using original feature values so the results can be described in business language. One-hot means are converted to percentages for neighbourhood, room type, and cancellation policy.

In [ ]:
# Start from original feature values; cap minimum nights for consistent reporting.
df_profile = df.copy()
df_profile["minimum nights"] = df_profile["minimum nights"].clip(upper=365)

category_groups = {
    "Neighbourhood": [c for c in df_profile.columns if c.startswith("neighbourhood group_")],
    "Room type": [c for c in df_profile.columns if c.startswith("room type_")],
    "Cancellation policy": [c for c in df_profile.columns if c.startswith("cancellation_policy_")]
}

profiles_k2 = {}

for variant_name in [
    "Current baseline",
    "Without coordinates",
    "Without host portfolio size"
]:
    # Fit labels were saved during the ablation experiment.
    labels = pd.Series(
        experiment_labels[(variant_name, 2)],
        index=df_profile.index,
        name="cluster"
    )
    temp = df_profile.copy()
    temp["cluster"] = labels

    # Use median as well as mean for features with long right tails.
    summary = temp.groupby("cluster").agg(
        listings=("price", "size"),
        price_mean=("price", "mean"),
        price_median=("price", "median"),
        minimum_nights_median=("minimum nights", "median"),
        reviews_mean=("number of reviews", "mean"),
        reviews_per_month_mean=("reviews per month", "mean"),
        host_listings_mean=("calculated host listings count", "mean"),
        host_listings_median=("calculated host listings count", "median"),
        availability_mean=("availability 365", "mean")
    )
    summary["share_pct"] = summary["listings"] / len(temp) * 100
    profiles_k2[variant_name] = summary.round(2)

    print(f"\n{'=' * 10} {variant_name} {'=' * 10}")
    display(profiles_k2[variant_name])

    for group_name, columns in category_groups.items():
        if columns:
            print(f"{group_name} distribution (%)")
            display((temp.groupby("cluster")[columns].mean() * 100).round(1))

### Profile interpretation

The current and no-coordinate k=2 profiles are nearly identical, even though removing coordinates improves silhouette. The no-coordinate solution divides the listings primarily into:

- **Large host portfolios / longer stays:** about 8.48% of listings; median host portfolio size 50; median minimum stay 30 days; average availability about 241 days; around 76.6% in Manhattan.
- **Smaller host portfolios / shorter stays:** about 91.52% of listings; median host portfolio size 1; median minimum stay 2 days; average availability about 139 days; neighbourhood mix mainly Brooklyn and Manhattan.

Average prices are close (approximately 518 vs. 525), so the candidate is not best described as a price-tier segmentation. These are descriptive summaries, not proof of host intent or demand.

## 7. Adjusted Rand Index: are the assignments sensitive to feature selection?

ARI compares partitions regardless of their numeric cluster labels. Values near 1 mean almost identical assignments; values near 0 indicate little agreement beyond chance; negative values indicate agreement below chance expectation.

This test supports whether the no-coordinate model retains the baseline partition and whether removing host portfolio size fundamentally changes it.

In [ ]:
ari_results = []

for k in [2, 3, 4]:
    baseline_labels = experiment_labels[("Current baseline", k)]

    for variant in ["Without coordinates", "Without host portfolio size"]:
        alternative_labels = experiment_labels[(variant, k)]
        ari_results.append({
            "k": k,
            "comparison": f"Baseline vs {variant}",
            "adjusted_rand_index": adjusted_rand_score(
                baseline_labels, alternative_labels
            )
        })

ari_df = pd.DataFrame(ari_results)
display(ari_df.round(4))

### ARI findings

The observed comparisons were approximately:

- `k=2`, baseline vs. without coordinates: **0.983** — almost identical assignments.
- `k=2`, baseline vs. without host portfolio size: approximately **-0.055** — very different assignments.
- `k=3`, baseline vs. without coordinates: 0.735.
- `k=4`, baseline vs. without coordinates: 0.306.

**Decision:** for the leading `k=2` model, removing coordinates simplifies the representation without materially changing the assignment. Removing host portfolio size fundamentally changes the partition, confirming that this feature drives an important part of the current segmentation.

## 8. Initialization stability for k=2, 3, and 4

K-Means can produce different solutions if its initialization changes. This cell repeats the fit with multiple seeds using the no-coordinate feature configuration. ARI is measured against the `random_state=42` partition for each k.

This tests **initialization stability**, not external validity or whether the clusters are inherently the right business segmentation.

In [ ]:
X_candidate = scaled_feature_sets["Without coordinates"]

# Test multiple seeds and store each model once.
seeds = [0, 21, 42, 7, 123]
reference_seed = 42
stability_labels = {}
stability_rows = {}

for k in [2, 3, 4]:
    for seed in seeds:
        model = KMeans(n_clusters=k, random_state=seed, n_init=10)
        labels = model.fit_predict(X_candidate)

        stability_labels[(k, seed)] = labels
        stability_rows[(k, seed)] = {
            "inertia": model.inertia_
        }

# Compare each seeded result with the same-k reference assignment.
stability_results = []
for k in [2, 3, 4]:
    reference_labels = stability_labels[(k, reference_seed)]

    for seed in seeds:
        labels = stability_labels[(k, seed)]
        stability_results.append({
            "k": k,
            "seed": seed,
            "ARI_vs_reference": adjusted_rand_score(reference_labels, labels),
            "inertia": stability_rows[(k, seed)]["inertia"]
        })

stability_df = pd.DataFrame(stability_results)
display(stability_df.round(4))

### Stability result and decision

In the saved experiment, k=2 had ARI values approximately 0.9994–1.0000 across seeds; k=3 and k=4 were also highly stable. Inertia varied only trivially across seeds.

This indicates that K-Means consistently finds the same partitions from different initializations. Combined with the k=2 silhouette of approximately 0.3912 and interpretable profiles, this supports retaining the no-coordinate `k=2` model as the leading candidate.

Stable clusters are not automatically meaningful clusters, so this evidence should be considered alongside the feature-ablation scores and business profiles.

## 9. Fit and evaluate the selected candidate

This section consolidates the chosen feature set and labels. It is the cleanest reference inside the rough-work notebook for the currently selected model. The concise `clustering.ipynb` reproduces this selected pipeline without the exploratory experiments.

In [ ]:
# Use the no-coordinate scaled matrix selected by the ablation experiment.
X_candidate = scaled_feature_sets["Without coordinates"]

# Fit the selected candidate deterministically.
final_model = KMeans(n_clusters=2, random_state=42, n_init=10)
final_labels = pd.Series(
    final_model.fit_predict(X_candidate),
    index=df.index,
    name="cluster"
)

# Check cluster sizes on the full dataset.
cluster_summary = (
    final_labels.value_counts()
    .sort_index()
    .rename("listings")
    .to_frame()
)
cluster_summary["share_pct"] = (
    cluster_summary["listings"] / len(final_labels) * 100
).round(2)

display(cluster_summary)

In [ ]:
# Describe the selected clusters using original, interpretable units.
df_final_profile = df.copy()
df_final_profile["minimum nights"] = (
    df_final_profile["minimum nights"].clip(upper=365)
)
df_final_profile["cluster"] = final_labels

final_profile = df_final_profile.groupby("cluster").agg(
    listings=("price", "size"),
    price_mean=("price", "mean"),
    price_median=("price", "median"),
    minimum_nights_median=("minimum nights", "median"),
    reviews_mean=("number of reviews", "mean"),
    reviews_per_month_mean=("reviews per month", "mean"),
    host_listings_mean=("calculated host listings count", "mean"),
    host_listings_median=("calculated host listings count", "median"),
    availability_mean=("availability 365", "mean")
).round(2)

final_profile["share_pct"] = (
    final_profile["listings"] / len(df_final_profile) * 100
).round(2)

display(final_profile)

# Summarize encoded categories as cluster-level percentages.
for group_name, columns in category_groups.items():
    if columns:
        print(f"\n{group_name} distribution (%) — selected candidate")
        display(
            (df_final_profile.groupby("cluster")[columns].mean() * 100)
            .round(1)
        )

## 10. Final silhouette diagnostics

This cell evaluates the selected model on the same reproducible 5,000-row sample. The cluster-level summary is useful because a single average silhouette can hide a poorly separated minority cluster.

In [ ]:
# Sample rows reproducibly while keeping the full-data model assignments.
final_sample_idx = X_candidate.sample(
    n=min(5000, len(X_candidate)),
    random_state=42
).index

X_eval_final = X_candidate.loc[final_sample_idx]
labels_eval_final = final_labels.loc[final_sample_idx]

selected_silhouette = silhouette_score(X_eval_final, labels_eval_final)
print(f"Selected candidate silhouette: {selected_silhouette:.4f}")

sample_scores = silhouette_samples(X_eval_final, labels_eval_final)
silhouette_by_cluster = pd.DataFrame({
    "cluster": labels_eval_final.to_numpy(),
    "silhouette": sample_scores
})

display(
    silhouette_by_cluster.groupby("cluster")["silhouette"]
    .agg(["mean", "median", "min", "max", "count"])
    .round(3)
)

# A negative score suggests an observation may fit the neighbouring cluster better.
negative_pct = (
    silhouette_by_cluster.groupby("cluster")["silhouette"]
    .apply(lambda values: values.lt(0).mean() * 100)
    .rename("negative_silhouette_pct")
)
display(negative_pct.round(2).to_frame())

### Selected model silhouette interpretation

The saved selected-model result was:
- Overall silhouette: **0.3912**.
- Cluster 0: mean silhouette ≈ 0.397; 4,586 sampled listings; no negative scores in this sample.
- Cluster 1: mean silhouette ≈ 0.329; 414 sampled listings; approximately 2.17% of its sampled listings had negative scores.

Interpretation: separation is moderate, and the smaller cluster is reasonably distinct, with a small fraction of boundary/overlapping observations. This supports an exploratory segmentation, not perfect separation.

## 11. PCA visualization and loadings for the selected feature set

PCA is used only to visualize and diagnose the selected scaled matrix. The model is still fitted on all selected scaled features, not on just the first two principal components.

The two-dimensional projection explained about **44.79%** of variance in the saved result. The first component was dominated by `calculated host listings count` (loading ≈ 0.938); the second was dominated by `reviews per month` (loading ≈ 0.895). This agrees with the profile finding that portfolio scale and review activity are key dimensions of the data.

In [ ]:
# Project the selected 22-feature matrix into two principal components for visualization.
pca_final = PCA(n_components=2, random_state=42)
X_pca_final = pca_final.fit_transform(X_candidate)

variance_explained = pca_final.explained_variance_ratio_.sum() * 100
print(f"Variance explained by PC1 and PC2: {variance_explained:.2f}%")

# Plot all listings, colored by selected cluster assignment.
plt.figure(figsize=(10, 7))
scatter = plt.scatter(
    X_pca_final[:, 0],
    X_pca_final[:, 1],
    c=final_labels.to_numpy(),
    s=8,
    alpha=0.45
)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("Selected Candidate: K-Means, k=2 (No Coordinates)")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.show()

# PCA loadings show the features most associated with each component direction.
pca_loadings_final = pd.DataFrame(
    pca_final.components_.T,
    index=X_candidate.columns,
    columns=["PC1", "PC2"]
)
pca_loadings_final["abs_PC1"] = pca_loadings_final["PC1"].abs()
pca_loadings_final["abs_PC2"] = pca_loadings_final["PC2"].abs()

print("Top contributors to PC1:")
display(pca_loadings_final.sort_values("abs_PC1", ascending=False).head(10))

print("Top contributors to PC2:")
display(pca_loadings_final.sort_values("abs_PC2", ascending=False).head(10))

## 12. Export selected candidate outputs

This is the same final export logic used by the concise pipeline notebook. The listing-level CSV preserves the input features and adds a `cluster` label. The profile CSV contains counts, shares, and interpretable numerical summaries for each cluster.

These outputs use the chosen **no-coordinate k=2 model**. Coordinates remain in the listing-level CSV because they were excluded only from the distance calculation, not deleted from the source data.

In [ ]:
# Write generated outputs separately from the input ML-ready dataset.
OUTPUT_DIR = Path("../data/ml/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Verify the cluster labels align one-to-one with the loaded listing rows.
if not final_labels.index.equals(df.index):
    raise ValueError("Cluster labels are not aligned with the source rows.")

# Preserve all original ML-ready columns and append the selected cluster label.
clustered_listings = df.copy()
clustered_listings["cluster"] = final_labels.to_numpy()

assignments_path = OUTPUT_DIR / "clustered_listings_kmeans_k2.csv"
clustered_listings.to_csv(assignments_path, index=False)

# Save a compact business profile table for the selected clusters.
profile_path = OUTPUT_DIR / "cluster_profile_kmeans_k2.csv"
final_profile.to_csv(profile_path, index=True)

# Confirm the output dimensions and final cluster distribution.
assert len(clustered_listings) == len(df)
assert clustered_listings["cluster"].notna().all()

print("Cluster assignments saved to:", assignments_path)
print("Cluster profiles saved to:", profile_path)
print("Exported listings:", len(clustered_listings))
display(
    (final_labels.value_counts(normalize=True).sort_index() * 100)
    .round(2)
    .rename("share_pct")
    .to_frame()
)

## 13. Summary: retained, changed, and not selected

### Retained in the selected candidate
- `k=2`, `random_state=42`, `n_init=10`.
- Log-transform `minimum nights`, `number of reviews`, and `calculated host listings count`.
- Cap `minimum nights` at 365 before transformation.
- Exclude `service fee` from the clustering matrix.
- Exclude latitude/longitude while retaining neighbourhood-group indicators.
- Robust-scale continuous features and leave binary/one-hot features as 0/1.
- Profile in original units and export both listing-level assignments and a summary table.

### Tested but not selected as the primary solution
- **Untransformed minimum nights:** yielded a misleadingly high silhouette alongside extremely tiny clusters; the first PCA component was almost entirely minimum nights.
- **Coordinates included:** nearly identical k=2 assignment (ARI ≈ 0.983) but a lower silhouette (0.350 vs. 0.3912).
- **Host portfolio size excluded:** more balanced cluster sizes but much lower silhouette (0.1682 at k=2); retain the feature provisionally because portfolio scale is business-relevant.
- **k=3 and k=4:** explored, but overall silhouette scores were much lower and some clusters had weak within-cluster separation.
- **PCA components as clustering inputs:** not used; PCA remains a visualization/diagnostic step.
- **NLP on `house_rules`, hierarchical clustering, and DBSCAN:** not implemented in this project stage; possible future work.

### Reproducibility note
The compact `clustering.ipynb` should be the notebook used to regenerate the two CSV outputs. This notebook intentionally contains more code and explanation so the model-selection reasoning remains transparent and reviewable.